# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begs44/flyrank-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The feature vector for the Refresh lane: one row = one page, label = impressions fell >20% (last 30d vs previous 30d), base rate 0.542. 19 numeric + 3 categorical features. Heavy-tailed traffic counts are log-scaled, `avg_position = 0` means "no data" so it becomes missing, missing word count / keyword data get their own 0/1 flag, numerics are median-filled and scaled, categoricals are one-hot encoded (missing = "unknown").

In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import roc_auc_score

SEED = 42
url = "https://raw.githubusercontent.com/begs44/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Label (Refresh lane): impressions fell >20%, last 30d vs previous 30d. Used to MEASURE only, never as an input.
df["target_decay"] = (df["trend_direction"] == "down").astype(int)
base = df["target_decay"].mean()

# ---- Feature vector ----
df["avg_position"] = df["avg_position"].replace(0, np.nan)            # 0 means "no data", not rank 0
df["has_word_count"] = df["word_count"].notna().astype(int)            # missingness becomes its own signal
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{c}"] = np.log1p(df[c])                                  # heavy right tails -> log scale
NUMERIC = ["search_volume", "competition", "cpc", "word_count",
           "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
           "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
           "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
           "has_word_count", "has_keyword_data"]
CATEGORICAL = ["content_type", "main_intent", "competition_level"]
df[CATEGORICAL] = df[CATEGORICAL].fillna("unknown")
FEATS = NUMERIC + CATEGORICAL
X, y, groups = df[FEATS], df["target_decay"], df["client_id"]

def preprocessor(num_cols):
    return make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),  # median fill, then scale
        (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))                         # unseen category -> all zeros

Xt = preprocessor(NUMERIC).fit_transform(X)
print(f"Rows: {len(df):,} pages | label base rate: {base:.3f}")
print(f"Feature vector: {len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical -> {Xt.shape[1]} model columns after one-hot")
print("Categories:", {c: sorted(df[c].unique()) for c in CATEGORICAL})

Rows: 30,000 pages | label base rate: 0.542
Feature vector: 19 numeric + 3 categorical -> 31 model columns after one-hot
Categories: {'content_type': ['comparison article', 'feedly article', 'keyword article'], 'main_intent': ['commercial', 'informational', 'navigational', 'transactional', 'unknown'], 'competition_level': ['HIGH', 'LOW', 'MEDIUM', 'unknown']}


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Every feature with its type, how much is missing, how missing is handled, and **when it is available**. Static metadata (keyword data, word count, age, content type) is known before any decision. The 90-day traffic features are the honest weak point: their window **overlaps** the label's last 30 days, so they partly describe decline already in progress. Section 3 measures how much of the skill comes from that overlap.


In [2]:
OVERLAP = "trailing 90d window: OVERLAPS the label's last 30d (tested in section 3)"
AVAILABLE_WHEN = {
    "search_volume": "keyword metadata, set when the page was planned",
    "competition": "keyword metadata, set when the page was planned",
    "cpc": "keyword metadata, set when the page was planned",
    "word_count": "fixed once the page is published",
    "log_impressions_90d": OVERLAP,
    "log_clicks_90d": OVERLAP,
    "log_sessions_90d": OVERLAP,
    "log_ai_sessions_90d": OVERLAP,
    "days_with_impressions": OVERLAP,
    "days_with_sessions": OVERLAP,
    "content_age_days": "known from the publish date",
    "days_since_last_update": "known from the last edit date",
    "ctr": "90d rate: overlaps, but a rate, not a volume change",
    "avg_position": "90d average: overlaps, but a level, not a volume change",
    "engagement_rate": "90d rate: overlaps",
    "scroll_rate": "90d rate: overlaps",
    "ai_traffic_pct": "90d rate: overlaps",
    "has_word_count": "known at publish",
    "has_keyword_data": "known when the page was planned",
    "content_type": "set at creation",
    "main_intent": "set at creation (keyword intent)",
    "competition_level": "keyword metadata, set when the page was planned",
}
FLAGGED = ("word_count", "search_volume")
notes = pd.DataFrame({
    "type": ["category" if c in CATEGORICAL else "numeric" for c in AVAILABLE_WHEN],
    "missing_%": [round(X[c].isna().mean() * 100, 1) for c in AVAILABLE_WHEN],
    "missing handled by": ["n/a ('unknown' level)" if c in CATEGORICAL else "0 -> missing, median fill" if c == "avg_position" else "median fill + flag" if c in FLAGGED
                           else "median fill" for c in AVAILABLE_WHEN],
    "available when?": list(AVAILABLE_WHEN.values()),
}, index=list(AVAILABLE_WHEN))
assert set(notes.index) == set(FEATS), "every feature needs a note"
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 220)
print(notes.to_string())

                            type  missing_%         missing handled by                                                           available when?
search_volume            numeric        8.2         median fill + flag                           keyword metadata, set when the page was planned
competition              numeric        8.2                median fill                           keyword metadata, set when the page was planned
cpc                      numeric        8.2                median fill                           keyword metadata, set when the page was planned
word_count               numeric       25.7         median fill + flag                                          fixed once the page is published
log_impressions_90d      numeric        0.0                median fill  trailing 90d window: OVERLAPS the label's last 30d (tested in section 3)
log_clicks_90d           numeric        0.0                median fill  trailing 90d window: OVERLAPS the label's last 30d (tested

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Four attacks on my own features, all scored out-of-fold with 5-fold cross-validation **grouped by client** (every client is scored by a model that never saw it). Base rate 0.542; random ranking = ROC-AUC 0.50.

1. **Label-derived column:** adding `trend_pct` (the label before the −20% threshold) pushes ROC-AUC from 0.66 to 0.998, near-perfect. Leak → removed.
2. **Sibling window:** adding `impressions_last_30d` (one half of the label formula) jumps to 0.87. Leak → removed.
3. **Overlapping windows:** with static metadata only, ROC-AUC is 0.53, barely above random. So most of the honest skill comes from 90-day traffic windows that overlap the label: the ranking **describes decline in progress, it does not forecast it**. Kept, and stated as a limitation.
4. **Split:** the same features under a random split score 0.71 vs 0.66 grouped. The gap is the model memorizing clients, so the grouped number is the one I report.

Single-feature scan: no feature ranks the label above 0.59 on its own → no hidden leak in the honest set.

In [3]:
def cv_auc(cols, split="grouped"):
    """Out-of-fold ROC-AUC, mean of 5 folds. Grouped = every client is scored by a model that never saw it."""
    num = [c for c in cols if c not in CATEGORICAL]
    Xc = df[cols]
    folds = (GroupKFold(5).split(Xc, y, groups) if split == "grouped"
             else StratifiedKFold(5, shuffle=True, random_state=SEED).split(Xc, y))
    scores = []
    for tr, te in folds:
        m = make_pipeline(preprocessor(num), LogisticRegression(max_iter=2000, C=0.5)).fit(Xc.iloc[tr], y.iloc[tr])
        scores.append(roc_auc_score(y.iloc[te], m.predict_proba(Xc.iloc[te])[:, 1]))
    return np.mean(scores)

print(f"Base rate {base:.3f} | random ranking = ROC-AUC 0.50\n")
print(f"Honest feature set, grouped by client        ROC-AUC = {cv_auc(FEATS):.3f}")

# Attack 1 - label-derived column: trend_pct IS the label before the -20% threshold
df["LEAK_trend_pct"] = df["trend_pct"].fillna(0)
print(f"+ trend_pct (label-derived)                  ROC-AUC = {cv_auc(FEATS + ['LEAK_trend_pct']):.3f}  <- leak, removed")
# Attack 2 - sibling window: last-30d impressions are one half of the label formula
df["LEAK_imp_last30"] = np.log1p(df["impressions_last_30d"])
print(f"+ impressions_last_30d (sibling window)      ROC-AUC = {cv_auc(FEATS + ['LEAK_imp_last30']):.3f}  <- leak, removed")
# Attack 3 - overlapping windows: how much skill is left without any traffic window?
STATIC = ["search_volume", "competition", "cpc", "word_count", "content_age_days", "days_since_last_update",
          "has_word_count", "has_keyword_data"] + CATEGORICAL
print(f"Static metadata only (no traffic windows)    ROC-AUC = {cv_auc(STATIC):.3f}  <- skill comes from overlapping windows")
# Attack 4 - split: a random split lets the model learn each client
print(f"Same honest features, RANDOM split           ROC-AUC = {cv_auc(FEATS, 'random'):.3f}  <- gap vs grouped = client memorization")
df.drop(columns=["LEAK_trend_pct", "LEAK_imp_last30"], inplace=True)

# Single-column scan: a feature that ranks the label almost perfectly on its own is a suspect
single = pd.Series({c: max(a := roc_auc_score(y, X[c].fillna(X[c].median())), 1 - a) for c in NUMERIC})
single = single.sort_values(ascending=False).round(3)
print("\nStrongest single features (ROC-AUC alone, direction ignored):\n" + single.head(5).to_string())
print("Suspects above 0.90:", list(single[single >= 0.9].index) or "none")

Base rate 0.542 | random ranking = ROC-AUC 0.50

Honest feature set, grouped by client        ROC-AUC = 0.661
+ trend_pct (label-derived)                  ROC-AUC = 0.998  <- leak, removed
+ impressions_last_30d (sibling window)      ROC-AUC = 0.873  <- leak, removed
Static metadata only (no traffic windows)    ROC-AUC = 0.530  <- skill comes from overlapping windows
Same honest features, RANDOM split           ROC-AUC = 0.710  <- gap vs grouped = client memorization

Strongest single features (ROC-AUC alone, direction ignored):
content_age_days         0.591
log_impressions_90d      0.584
days_with_impressions    0.579
search_volume            0.560
has_keyword_data         0.540
Suspects above 0.90: none


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Every field I refused to use, with one line of why. The code below also **guards** the feature list (it fails if a banned column sneaks in) and runs a **privacy scan**: no emails, URLs or domains in any text column, and the client and content IDs are pseudonymous hashes only.

In [4]:
EXCLUDED = {
    "trend_direction": "the label itself",
    "trend_pct": "the label before thresholding (near-perfect score in section 3)",
    "impressions_last_30d / impressions_prev_30d": "the two halves of the label formula",
    "clicks_*, sessions_* (_last_30d, _prev_30d)": "same windows as the label: sibling leak",
    "impression_tier": "bucket of the overlapping impression volume (pre-made bucket)",
    "position / age / freshness / word / char tiers, age_tier_order": "pre-made buckets (existing-system choices); raw values used instead",
    "content_id": "pseudonymous ID with no meaning; would memorize pages",
    "client_id": "used ONLY to group the folds, never as a feature",
    "provider_used / model_used": "production metadata about how text was generated, not a page signal",
    "char_count": "near-duplicate of word_count",
    "pageviews / users / engaged_sessions / scroll_events (_90d)": "near-duplicates of sessions_90d",
}
for k, v in EXCLUDED.items():
    print(f"- {k:<62} {v}")

# Guard: no excluded column may reach the model
banned = ["trend_direction", "trend_pct", "content_id", "client_id", "impression_tier", "position_tier", "age_tier",
          "age_tier_order", "freshness_tier", "word_count_tier", "char_count_tier", "provider_used", "model_used"]
bad = [c for c in FEATS if c in banned or c.endswith(("_last_30d", "_prev_30d"))]
assert not bad, bad
print("\nLeakage guard passed: no label, label-window, tier or ID column in the feature vector.")

# Privacy check: text columns should hold only pseudonyms or categories - no emails, URLs or domains
txt = df.select_dtypes("object")
pat = r"@|https?://|www\.|\.(?:com|net|org|io)\b"
hits = {c: int(txt[c].astype(str).str.contains(pat, regex=True).sum()) for c in txt}
print("Privacy scan (emails / URLs / domains):", {c: n for c, n in hits.items() if n} or "none found in any text column")
ids_ok = df["content_id"].str.fullmatch(r"content_[0-9a-f]+").all() and df["client_id"].str.fullmatch(r"client_[0-9a-f]+").all()
print("content_id and client_id are pseudonymous hashes only:", bool(ids_ok))

- trend_direction                                                the label itself
- trend_pct                                                      the label before thresholding (near-perfect score in section 3)
- impressions_last_30d / impressions_prev_30d                    the two halves of the label formula
- clicks_*, sessions_* (_last_30d, _prev_30d)                    same windows as the label: sibling leak
- impression_tier                                                bucket of the overlapping impression volume (pre-made bucket)
- position / age / freshness / word / char tiers, age_tier_order pre-made buckets (existing-system choices); raw values used instead
- content_id                                                     pseudonymous ID with no meaning; would memorize pages
- client_id                                                      used ONLY to group the folds, never as a feature
- provider_used / model_used                                     production metadata about

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.